# Book-Layout RT-DETR: Cross-Page Layout Semantic Fusion Pipeline
Pipeline kết hợp **RT-DETRv2** và cơ chế **Book Layout Semantic Fusion** (tương tự Multimodal OCR Fusion nhưng áp dụng cho ngữ nghĩa bố cục toàn cuốn sách).

- **Pipeline 1 (Frozen Scout):** RT-DETRv2 quét qua các trang trong cuốn sách để trích xuất đặc trưng bố cục.
- **Cầu nối (Fusion Module):** Nén ngữ nghĩa toàn bộ cuốn sách thành tập vector $E_{\text{book}} \in \mathbb{R}^{K \times 256}$.
- **Pipeline 2 (Context-Aware RT-DETRv2):** Các Object Queries của RT-DETR được hòa trộn thông tin với $E_{\text{book}}$ thông qua **Cross-Attention Fusion** trước khi đưa ra nhãn và tọa độ box cuối cùng.

In [ ]:
import os
import sys
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# Thiết lập đường dẫn thư mục gốc
ROOT_DIR = Path.cwd().parents[1] if "notebooks" in Path.cwd().parts else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

METHOD_DIR = ROOT_DIR / "methods" / "method_rtdetr_fusion"
print(f"Workspace Root: {ROOT_DIR}")
print(f"Method Directory: {METHOD_DIR}")

## 1. Lịch sử huấn luyện & Biểu đồ Loss / Accuracy
Đọc dữ liệu lịch sử fine-tuning từ `training_history.csv` và hiển thị đồ thị quá trình huấn luyện.

In [ ]:
csv_path = METHOD_DIR / "report" / "training_history.csv"
if csv_path.exists():
    df_history = pd.read_csv(csv_path)
    display(df_history)
else:
    print("Chưa tìm thấy file training_history.csv")

In [ ]:
plot_path = METHOD_DIR / "report" / "training_report.png"
if plot_path.exists():
    img = Image.open(plot_path)
    plt.figure(figsize=(15, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Training Metrics Report (Loss & Validation Accuracy)", fontsize=14, fontweight="bold")
    plt.show()
else:
    print("Chưa tìm thấy file training_report.png")

## 2. Bảng Đối chiếu Benchmark: Base RT-DETRv2 vs Book-Layout RT-DETR
So sánh chi tiết trên từng lớp (Class-level Breakdown) giữa mô hình đơn trang (Base) và mô hình có Semantic Fusion toàn cuốn sách.

In [ ]:
benchmark_json = METHOD_DIR / "report" / "benchmark_comparison.json"
if benchmark_json.exists():
    with open(benchmark_json, "r", encoding="utf-8") as f:
        bench_data = json.load(f)
    
    df_bench = pd.DataFrame(bench_data["classes"])
    # Định dạng phần trăm
    for col in df_bench.columns:
        if col != "class":
            df_bench[col] = (df_bench[col] * 100).round(2).astype(str) + "%"
    
    print("\n=== KẾT QUẢ SO SÁNH TRÊN TỪNG CLASS ===")
    display(df_bench)
    
    print("\n=== TỔNG THỂ MICRO-AVERAGE ===")
    print(f"- Base RT-DETR Micro F1:   {bench_data['overall']['base_micro_f1'] * 100:.2f}%")
    print(f"- Fusion RT-DETR Micro F1: {bench_data['overall']['fusion_micro_f1'] * 100:.2f}%")
else:
    print("Chưa có file benchmark_comparison.json")

## 3. Chạy Huấn luyện Bổ sung hoặc Đánh giá Quy mô lớn
Bạn có thể chạy lệnh bên dưới để fine-tuning toàn diện hoặc đánh giá trên toàn bộ tập test.

In [ ]:
# Lệnh huấn luyện mở rộng:
# !py -3.12 methods/method_rtdetr_fusion/train.py --epochs 10 --limit-train-pages 500 --batch-size 2

# Lệnh đánh giá so sánh:
# !py -3.12 methods/method_rtdetr_fusion/evaluate.py --limit-pages 100